In [15]:
import os
import pickle
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [ ]:

video_path = r"C:\Users\User\Desktop\deeplearning\fusion\video_features_256.pkl"
audio_path = r"C:\Users\User\Desktop\deeplearning\fusion\audio_feat.pkl"
text_path = r"C:\Users\User\Desktop\deeplearning\fusion\text_features_256(basic+earlystop).pkl"

# 2199개 샘플, 이미지 높이 224, 이미지 너비 224, RGB 3채널
with open(video_path, "rb") as f: 
    video_data = pickle.load(f)
X_video = video_data["features"]
X_video_labels = video_data["labels"] 



with open(audio_path, "rb") as f:
    audio_data = pickle.load(f)
X_audio = audio_data["audio_feat"]
X_audio_labels = audio_data["labels"]


text_data = torch.load(text_path,map_location=device)
X_text = text_data["features"]
X_text_labels = text_data["labels"]

# with open(text_path, "rb") as f:
#     text_data = pickle.load(f)
# X_text = text_data["features"]
# X_text_labels = text_data["labels"]

X_video_labels = torch.as_tensor(X_video_labels).long()
X_audio_labels = torch.as_tensor(X_audio_labels).long()
X_text_labels = torch.as_tensor(X_text_labels).long()


print(f"X_video      : {X_video.shape} ") # (224, 224, 3)
print(f"X_video_labels: (긍정 {X_video_labels.sum()} / 부정 {(X_video_labels==0).sum()})")

print(f"X_audio      : {X_audio.shape} ")
print(f"X_audio_labels: (긍정 {X_audio_labels.sum()} / 부정 {(X_audio_labels==0).sum()})")

print(f"X_text      : {X_text.shape} ")
print(f"X_text_labels: (긍정 {X_text_labels.sum()} / 부정 {(X_text_labels==0).sum()})")


X_video      : torch.Size([2199, 256]) 
X_video_labels: (긍정 1176 / 부정 1023)
X_audio      : torch.Size([2199, 256]) 
X_audio_labels: (긍정 1080 / 부정 1119)
X_text      : torch.Size([2199, 256]) 
X_text_labels: (긍정 1080 / 부정 1119)


그러나 라벨 분포가 video만 다르다

In [22]:
X_video_labels = torch.as_tensor(X_video_labels).cpu()
X_audio_labels = torch.as_tensor(X_audio_labels).cpu()
X_text_labels = torch.as_tensor(X_text_labels).cpu()

print(torch.equal(X_audio_labels, X_text_labels))
print(torch.equal(X_video_labels, X_text_labels))
print(torch.equal(X_video_labels, X_audio_labels))

True
False
False


feature 차원 기준 concat은 가능하나 label 기준 video 라벨이 달라 확인 필요
각자 차원수 맞춰왔으니 각 모달리티별 인코더 생략 가능

In [27]:
import torch.nn as nn

X_video = X_video.to(device)
X_audio = X_audio.to(device)
X_text = X_text.to(device)

# 세 모달리티 concat → MLP
fused = torch.cat([X_video, X_audio, X_text], dim=1)  # (N, 768)

mlp = nn.Sequential(
    nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 2)
)

y = torch.as_tensor(X_text_labels).long().to(device)  # (N,)

print(fused.shape)  # (N, 768)
print(y.shape)
print(fused.device)

torch.Size([2199, 768])
torch.Size([2199])
cuda:0


In [28]:
# train / val / test split
N = fused.shape[0]
indices = torch.randperm(N,device=device)

train_size = int(0.7 * N)
val_size = int(0.15 * N)

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

X_train, y_train = fused[train_idx], y[train_idx]
X_val, y_val = fused[val_idx], y[val_idx]
X_test, y_test = fused[test_idx], y[test_idx]

In [30]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score

In [ ]:
#DataLoader
batch_size = 32

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=32,
    shuffle=False
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=32,
    shuffle=False
)


In [37]:
MLP = nn.Sequential(
    nn.Linear(768, 256), 
    nn.ReLU(), 
    nn.Dropout(0.3),

    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(64, 2)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(MLP.parameters(), lr=1e-3)


In [57]:
def evaluate(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item() 

            _, preds = torch.max(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    return avg_loss, acc, prec, rec, f1

In [58]:
epochs = 20

for epoch in range(epochs):
    mlp.train()
    total_loss = 0

for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)

    optimizer.zero_grad()

    outputs = mlp(X_batch)
    loss = criterion(outputs, y_batch)

    loss.backward()
    optimizer.step()

In [59]:
print(next(mlp.parameters()).device)
print(fused.device)

cuda:0
cuda:0


In [60]:
mlp = mlp.to(fused.device)
optimizer = optim.Adam(mlp.parameters(), lr=1e-3)

print(next(mlp.parameters()).device)
print(fused.device)

cuda:0
cuda:0


In [66]:
epochs = 20


for epoch in range(epochs):
    mlp.train()
    
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()

        outputs = mlp(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    val_loss, val_acc, val_prec, val_rec, val_f1 = evaluate(mlp, val_loader)

    # print(f"Epoch {epoch+1}/{epochs} ")
    # print(f"Loss: {train_loss:.4f} ")
    # print(f"Val Acc: {val_acc:.4f} ")
    # print(f"Val Prec: {val_prec:.4f} ")
    # print(f"Val Rec: {val_rec:.4f} ")
    # print(f"Val F1: {val_f1:.4f}")

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f} - Val Prec: {val_prec:.4f} - Val Rec: {val_rec:.4f} - Val F1: {val_f1:.4f}")

test_loss, test_acc, test_prec, test_rec, test_f1 = evaluate(mlp, test_loader)

print("\n=====결과=======")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc: {test_acc:.4f} ")
print(f"Test Prec: {test_prec:.4f} ")
print(f"Test Rec: {test_rec:.4f} ")
print(f"Test F1: {test_f1:.4f}")

Epoch 1/20 - Train Loss: 0.2091 - Val Loss: 0.2976 - Val Acc: 0.9119 - Val Prec: 0.9032 - Val Rec: 0.9091 - Val F1: 0.9061
Epoch 2/20 - Train Loss: 0.1869 - Val Loss: 0.2766 - Val Acc: 0.9088 - Val Prec: 0.8974 - Val Rec: 0.9091 - Val F1: 0.9032
Epoch 3/20 - Train Loss: 0.1760 - Val Loss: 0.2912 - Val Acc: 0.9149 - Val Prec: 0.9145 - Val Rec: 0.9026 - Val F1: 0.9085
Epoch 4/20 - Train Loss: 0.1765 - Val Loss: 0.3057 - Val Acc: 0.9088 - Val Prec: 0.8827 - Val Rec: 0.9286 - Val F1: 0.9051
Epoch 5/20 - Train Loss: 0.1834 - Val Loss: 0.3060 - Val Acc: 0.9088 - Val Prec: 0.8974 - Val Rec: 0.9091 - Val F1: 0.9032
Epoch 6/20 - Train Loss: 0.1766 - Val Loss: 0.2943 - Val Acc: 0.9088 - Val Prec: 0.8827 - Val Rec: 0.9286 - Val F1: 0.9051
Epoch 7/20 - Train Loss: 0.1732 - Val Loss: 0.3278 - Val Acc: 0.9088 - Val Prec: 0.9026 - Val Rec: 0.9026 - Val F1: 0.9026
Epoch 8/20 - Train Loss: 0.1738 - Val Loss: 0.3016 - Val Acc: 0.9088 - Val Prec: 0.8974 - Val Rec: 0.9091 - Val F1: 0.9032
Epoch 9/20 - Tra